# Basic Data Preprocessing of the PHM Challenge Milling Dataset

In [ ]:
# Imports
import scipy.io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive, fixed, interact_manual, widgets
import os
import glob
from tqdm.notebook import tqdm

#### Load csv with pandas and display it

In [ ]:
# data_folder = '/Volumes/FBK-NAS/01_Tool_WearDatasets/02_PHM_data_challenge_2010/raw_data'
data_folder = '/Users/flore/Documents/01_Forschung/09_Coding.nosync/00_DATASETS/01_Tool_WearDatasets/02_PHM data challenge 2010'
cutters_with_wear = ['c1', 'c4', 'c6']

wear_data_frames = []
for cutter in cutters_with_wear:
    wear_file_path = os.path.join(data_folder, cutter, f'{cutter}_wear.csv')
    try:
        df = pd.read_csv(wear_file_path)
        df['cutter'] = cutter
        # The wear file has columns: 'cut', 'flute_1', 'flute_2', 'flute_3'
        df.rename(columns={'flute_1': 'flute1', 'flute_2': 'flute2', 'flute_3': 'flute3'}, inplace=True)
        df['cut_ID'] = df['cutter'] + '_' + df['cut'].astype(str)
        wear_data_frames.append(df)
    except FileNotFoundError:
        print(f"Warning: Wear file not found for {cutter} at {wear_file_path}")

if wear_data_frames:
    wear_df = pd.concat(wear_data_frames, ignore_index=True)
    # Reorder columns to match the request
    wear_df = wear_df[['cutter', 'cut', 'cut_ID', 'flute1', 'flute2', 'flute3']]
    print("wear_df created successfully.")
    display(wear_df.head())
else:
    print("No wear data was loaded. wear_df is empty.")
    wear_df = pd.DataFrame(columns=['cutter', 'cut', 'cut_ID', 'flute1', 'flute2', 'flute3'])


In [ ]:
wear_df['cutter'].unique()

In [ ]:
# Define column names for the signal files
signal_col_names = [
    'Fx', 'Fy', 'Fz',  # Force in X, Y, Z
    'Vx', 'Vy', 'Vz',  # Vibration in X, Y, Z
    'AE_RMS'           # AE-RMS (V)
]

all_cuts_data = []

# Load signals only for cutters with wear data
cutters_with_wear = ['c1', 'c4', 'c6']

for cutter in cutters_with_wear:
    cutter_number = cutter[1:]  # Extracts the number, e.g., '1' from 'c1'
    
    # The signal files are in a nested directory, e.g., c1/c1/
    signal_files_dir = os.path.join(data_folder, cutter, cutter)
    # Create a glob pattern to find all signal files for the cutter
    glob_pattern = os.path.join(signal_files_dir, f'c_{cutter_number}_*.csv')
    signal_files = sorted(glob.glob(glob_pattern))

    if not signal_files:
        print(f"Warning: No signal files found for {cutter} in {signal_files_dir}")
        continue
    
    print(f"\nProcessing cutter: {cutter}...")
    for file_path in tqdm(signal_files, desc=f"Loading {cutter} signals"):
        filename = os.path.basename(file_path)
        
        # Extract the cut number from the filename (e.g., 'c_1_001.csv' -> 1)
        try:
            cut_number = int(filename.split('_')[-1].replace('.csv', ''))
        except (ValueError, IndexError):
            print(f"Warning: Could not parse cut number from filename: {filename}")
            continue

        # Create the unique ID for the cut
        id_val = f"{cutter}_{cut_number}"
        
        # Read the signal data, which has no header
        try:
            # Read the raw data into a NumPy array for efficiency
            signal_data = np.loadtxt(file_path, delimiter=',')
            
            # Create a dictionary for the current cut
            cut_data = {
                'cut_ID': id_val,
                'cutter': cutter,
                'cut': cut_number
            }
            
            # Add each signal as a separate column with the NumPy array
            for i, col_name in enumerate(signal_col_names):
                cut_data[col_name] = signal_data[:, i]
                
            all_cuts_data.append(cut_data)
            
        except Exception as e:
            print(f"Error processing file {file_path}: {e}")

if all_cuts_data:
    # Create the DataFrame from the list of dictionaries
    signals_DF = pd.DataFrame(all_cuts_data)
    
    # Reorder columns for clarity
    ordered_cols = ['cutter', 'cut', 'cut_ID'] + signal_col_names
    signals_DF = signals_DF[ordered_cols]
    
    print("\nsignals_DF with nested arrays created successfully.")
    display(signals_DF.head())
    
    # Verify the structure by checking the type of a signal column
    print("\nData type of a signal column entry:")
    print(type(signals_DF['Fx'].iloc[0]))
else:
    print("\nNo signal data was loaded. signals_DF is empty.")
    signals_DF = pd.DataFrame(columns=['cutter', 'cut', 'cut_ID'] + signal_col_names)


In [ ]:
# Merge the wear data with the signals data
milling_df = pd.merge(signals_DF, wear_df, on='cut_ID', how='left', suffixes=('', '_wear'))

# The merge might create duplicate 'cutter' and 'cut' columns. Let's clean them up.
# We can drop the ones from the wear_df side.
milling_df.drop(columns=['cutter_wear', 'cut_wear'], inplace=True, errors='ignore')

#rename cutter to case_ID and cut to run and cut_ID to ExperimentIndex
milling_df.rename(columns={'cutter': 'CaseID', 'cut': 'run', 'cut_ID': 'ExperimentIndex'}, inplace=True)

In [ ]:
# Rename the entries of CaseID to c1 = cutter_1, and so on
milling_df['CaseID'] = milling_df['CaseID'].replace({'c1': 'tool_1', 'c4': 'tool_4', 'c6': 'tool_6'})

In [ ]:
milling_df.head()

In [ ]:
## Homogenization of the dataset
# Clalculate the force magnitude and vibration magnitude and add them as new columns
def calculate_magnitude(row, components):
	# Sum of squares of component arrays
	sum_sq = np.sum([row[c]**2 for c in components], axis=0)
	return np.sqrt(sum_sq)

milling_df['F_mag'] = milling_df.apply(calculate_magnitude, components=['Fx', 'Fy', 'Fz'], axis=1)
milling_df['V_mag'] = milling_df.apply(calculate_magnitude, components=['Vx', 'Vy', 'Vz'], axis=1)
# Now we built a normalized ewar column for each flute and a combined one
flutes = ['flute1', 'flute2', 'flute3']
for flute in flutes:
    min_wear = milling_df[flute].min()
    max_wear = milling_df[flute].max()
    # No wear = 1 and max wear = 0, where a higher number means less wear
    milling_df[f'{flute}_norm'] = (milling_df[flute] - min_wear) / (max_wear - min_wear)
# Create a combined normalized wear column by choosing the max of the normalized wear of all flutes
milling_df['wear_norm'] = milling_df[[f'{flute}_norm' for flute in flutes]].max(axis=1)
milling_df.head()

In [ ]:
# Generate a smaller version of the dataframe by removing Fx, Fy, Fz, Vx, Vy, Vz,flute1_norm, flute2_norm, flute3_norm,flute1, flute2, flute3, F_mag
milling_df_small = milling_df.drop(columns=['Fx', 'Fy', 'Fz', 'Vx', 'Vy', 'Vz', 'flute1_norm', 'flute2_norm', 'flute3_norm', 'flute1', 'flute2', 'flute3', 'F_mag'])
# Generate a homogenized data file for comparison with the other datasets
## Rename AE_RMS to AE, V_mag to Vib_Spindle
milling_df_small = milling_df_small.rename(columns={'AE_RMS': 'AE', 'V_mag': 'Vib_Spindle'})
milling_df_small.head()

In [ ]:
milling_df_small = milling_df_small.rename(columns={'AE': 'AE_spindle', 'Vib_Spindle': 'vib_spindle'})

In [ ]:
milling_df_small.head()

In [ ]:
import seaborn as sns
# Visualize for each cutter over time ('cut') the wear_norm
plt.figure(figsize=(12, 6))
sns.lineplot(data=milling_df_small, x='run', y='wear_norm', hue='CaseID', marker='o')
plt.title('Normalized Tool Wear Over Time for Each Cutter')
plt.xlabel('Cut Number')
plt.ylabel('Normalized Wear (1 = No Wear, 0 = Max Wear)')
plt.legend(title='Cutter ID', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
# Save the final, merged DataFrame to a single Parquet file
#milling_df.to_parquet('phm2007_milling_normalized.parquet')
milling_df_small.to_parquet('phm2007_milling_normalized_small.parquet')
print("Merged DataFrame 'milling_df' created and saved to 'phm2007_milling_normalized.parquet'.")
display(milling_df_small.head())


In [ ]:
milling_df_small['CaseID'].unique()

In [ ]:
# Typical signal length
lengths = milling_df_small['vib_spindle'].apply(len)
print(f"Typical signal length: {lengths.mode()[0]} samples")

In [ ]:
# Plot some samples of the vibration signals for different wear states
def plot_vibration_signals(df, case_id, num_samples=3):
    case_data = df[df['CaseID'] == case_id]
    sorted_data = case_data.sort_values(by='wear_norm')
    
    # Select samples evenly spaced across the wear range
    indices = np.linspace(0, len(sorted_data) - 1, num_samples, dtype=int)
    selected_samples = sorted_data.iloc[indices]
    
    plt.figure(figsize=(15, 8))
    
    for i, (_, row) in enumerate(selected_samples.iterrows()):
        plt.subplot(num_samples, 1, i + 1)
        plt.plot(row['vib_spindle'], label=f'Run: {row["run"]}, Wear: {row["wear_norm"]:.2f}')
        plt.title(f'Vibration Signal - {case_id} - Run {row["run"]}')
        plt.xlabel('Sample Index')
        plt.ylabel('Vibration Amplitude')
        plt.legend()
        plt.grid()
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_vibration_signals(milling_df_small, 'tool_1', num_samples=3)